# KG intent chat session -> turns + knowledge graph

An intent-driven sibling of **[`kg_chat_session.ipynb`](kg_chat_session.ipynb)**: same per-turn
flow (extract with `LLM_EventExtraction`, push with `populate_ekg_from_annotations()`, look for a
knowledge-graph gap, turn it into a follow-up question with `LLMTripleReplier`, otherwise fall
back to the default LLM reply), but the *gap-finding* step is different.

`kg_chat_session.ipynb` uses `kg_gap_finder.py`, which derives "what's expected" from peer
statistics: a gap only fires once a MAJORITY of an activity's own peers (other instances of the
same type already in the graph) share the predicate in question. That means the very FIRST
`take_food` activity ever pushed to the graph can never produce a gap -- it has no peers yet.

This notebook uses **`KgIntentChatSession`** (from `chat_sessions.py`) instead, which reads
"what's expected" straight from hand-authored **intents** -- one JSON file per topic under
**[`intents/`](../intents/)** at the project root (`diet_intents.json`, `condition_intents.json`,
`excercise_intents.json`, `medication_intents.json`, `symptom_intents.json`), each covering one
or more `data_type.ActivityType` values. See
**[`chat_from_kg/intent_gap_finder.py`](../src/cltl/chat_from_kg/intent_gap_finder.py)** for the
exact schema and priority order, but in short, for an eaten/drunk activity:

1. **`patient_type`** -- what was eaten/drunk (a `patient` of type `food`/`drink`) -- asked
   about first.
2. **`activity_date`** -- when -- asked about next, but only once (1) is filled in.
3. **`secondary_objectives.patient_qualification`** -- how much -- asked about last, only once
   (1) and (2) are both filled in.

Each requirement must be met before the next one is even considered -- unlike
`kg_chat_session.ipynb`'s gaps, which are all found (and asked about, most-affected-first) in one
go. **If an activity's own type has no matching intent at all**, `KgIntentChatSession` never asks
an intent-driven question about it -- the agent's reply for it always falls back to the plain LLM
reply, exactly like `kg_chat_session.ipynb` does when it simply has no gap left to ask about.

**Before running this:** same requirements as `kg_chat_session.ipynb` -- `OPENAI_API_KEY` set,
and a reachable knowledge-graph SPARQL endpoint (e.g. a local GraphDB `event_sandbox`
repository) at `KG_ADDRESS` below.

In [1]:
import time

from chat_sessions import KgIntentChatSession, save_turns

KG_ADDRESS = "http://localhost:7200/repositories/event_sandbox"
KG_LOG_DIR = "kg_logs"
HUMAN = "Mehmet"
HUMAN = "Jan"
# Directory of intent *.json files -- None auto-discovers the project's own intents/ folder (see
# intent_gap_finder.load_intents()/_default_intents_dir()), which works from this notebook's own
# directory. Point it elsewhere to try a different set of intents without editing the project's own.
INTENTS_DIR = None

# A FRESH id every run, not a fixed constant -- same reasoning as kg_chat_session.ipynb's own
# CHAT_ID cell: reusing a fixed id across separate runs makes every run's activities collide on
# the same subject URIs, corrupting the graph. See that notebook's markdown for the full story.
CHAT_ID = int(time.time())


## Run a live, intent-driven chat

Same window as `kg_chat_session.ipynb` (`kg_chat_gui.py`'s `ChatWindow`): the transcript scrolls
on the left, and -- since `KG_ADDRESS` points at a GraphDB repository -- a graph panel on the
right shows the activity currently being discussed. There is no "Gap sensitivity" slider here:
that control only appears for a session with a `gap_threshold` (peer-vote sensitivity), which
`KgIntentChatSession` deliberately has none of -- an intent's requirements are fixed, not a
majority-vote threshold to tune.

Each agent turn is labeled **[KG]** when it came from an intent-driven follow-up question, or
**[LLM]** when it's the default LLM reply (no matching intent, or nothing left for the matching
one to ask about) -- read straight from `kg_session.reply_sources`, same as
`kg_chat_session.ipynb`.

**While it's running**, each turn prints a diagnostic block to this cell's own output: what it
pushed to the knowledge graph, which intent (if any) matched the activity just mentioned, and
which requirement its reply was actually about -- see `kg_session.turn_log` below.

**To stop:** click **Quit**, or type "quit"/"bye"/... The window closes and the cell finishes,
and the conversation, a statistics summary, and this per-turn log are written to three
timestamped JSON files under `notebooks/chat_logs/` (`chat<chat>_turns_<stamp>.json` /
`chat<chat>_stats_<stamp>.json` / `chat<chat>_gaplog_<stamp>.json`) -- their paths are printed
below the cell. `run_gui()` then returns `kg_session.turns`, so everything below still works
unchanged; pass `save_dir=None` to skip the automatic save.

In [ ]:
from kg_chat_gui import run_gui

kg_session = KgIntentChatSession(
    chat=CHAT_ID,
    human=HUMAN,
    kg_address=KG_ADDRESS,
    log_dir=KG_LOG_DIR,
    intents_dir=INTENTS_DIR,
)
print(f"Loaded {len(kg_session.intents)} intent(s) covering activity types: "
      f"{sorted({t for intent in kg_session.intents for t in intent.get('activity_types', [])})}")
kg_turns = run_gui(kg_session)

Loaded 15 intent(s) covering activity types: ['economic_condition', 'exercise', 'measurement', 'mental_condition', 'physical_condition', 'sleep', 'social_condition', 'symptom', 'take_drink', 'take_food', 'take_medicine']


Inspect what was extracted, pushed, and where each agent reply came from:

In [1]:
print(f"{len(kg_session.turns)} turns, {len(kg_session.annotations)} annotated, "
      f"{len(kg_session.kg_pushes)} pushes to the knowledge graph")
print("reply sources:", kg_session.reply_sources)
kg_session.kg_pushes

NameError: name 'kg_session' is not defined

`kg_session.turn_log` has the same per-turn breakdown that was printed live above, for each turn:
what it pushed, which intent (if any) matched, and which requirement its reply was about (`None`
for a default, non-intent-driven reply):

In [4]:
kg_session.turn_log

[{'turn': 1,
  'speaker': 'Mehmet',
  'utterance': 'I cycled for an hour yesterday',
  'triples_pushed': [{'subject': 'cycled',
    'predicate': 'agent_patient',
    'object': 'I'},
   {'subject': 'cycled', 'predicate': 'time', 'object': 'for an hour'},
   {'subject': 'cycled', 'predicate': 'time', 'object': 'yesterday'}],
  'gap_queries': [{'subject': 'http://cltl.nl/leolani/n2mu/chat1789486133.1',
    'activity_type': 'exercise',
    'intent_source': 'excercise_intents.json',
    'after_dedup': 1}],
  'selected_gap': {'subject': 'http://cltl.nl/leolani/n2mu/chat1789486133.1',
   'predicate': 'duration',
   'kind': 'predicate',
   'peer_coverage': None}},
 {'turn': 2,
  'speaker': 'agent',
  'utterance': 'How long did you cycle yesterday?',
  'triples_pushed': [{'subject': 'cycle',
    'predicate': 'agent_patient',
    'object': 'you'},
   {'subject': 'cycle', 'predicate': 'time', 'object': 'yesterday'}],
  'gap_queries': [],
  'selected_gap': None},
 {'turn': 3,
  'speaker': 'Mehmet'

In [5]:
save_turns(kg_session.turns, "kg_intent_turns.json")

Wrote 12 turns to kg_intent_turns.json
